In [1]:
!pip install transformers==4.56.1 torch==2.8.0+cu126
!pip install huggingface-hub==0.35.0 datasets== 4.0.0

  Using cached transformers-4.56.1-py3-none-any.whl.metadata (42 kB)


ERROR: Could not find a version that satisfies the requirement torch==2.8.0+cu126 (from versions: 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0, 2.14.0, 2.14.1)

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for torch==2.8.0+cu126

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: 'datasets==': Expected semicolon (after name with no version specifier) or end
    datasets==
            ^


In [2]:
pip install pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install seaborn tabulate


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer
import torch
import gzip, json, random, time
import seaborn as sns
from tabulate import tabulate

In [5]:
# Global variable to store AI models
ai_models = {}
def setup_ai_models():
    """Setup available AI models"""
    global ai_models
    print("Loading AI models for demonstration...")
    ai_models['text_generator'] = pipeline(
        "text-generation",
        model="gpt2",
        max_length=100)

    ai_models['sentiment'] = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english" )

    ai_models['classifier'] = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

    print("✅ Models loaded successfully!")
    return ai_models
# Setup models and run demos

ai_models = setup_ai_models()

Loading AI models for demonstration...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

✅ Models loaded successfully!


In [6]:
def demonstrate_text_generation():
    """Show how AI generates text"""
    print("\nTEXT GENERATION DEMO")
    print("-" * 50)
    prompts = ["The future of artificial intelligence","Best practices for online shopping","Benefits of using AI in business"]
    if 'text_generator' in ai_models:
        for prompt in prompts:
            try:
                result = ai_models['text_generator'](prompt,max_length=60,num_return_sequences=1,temperature=0.7)
                generated = result[0]['generated_text']
                print(f"Prompt: {prompt}")
                print("AI Response:")
                print(generated)
                print("-" * 50)
            except Exception as e:
                print(f"Error with prompt: {prompt}")
                print(f"Details: {e}")
                print("-" * 50)
# Run the demo
demonstrate_text_generation()

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'temperature', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



TEXT GENERATION DEMO
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Prompt: The future of artificial intelligence
AI Response:
The future of artificial intelligence is very uncertain."

The research comes at a time when AI has a growing presence in the workplace, which has led to its emergence as a key tool for the future of artificial intelligence in both the scientific and the legal sectors.

Image copyright BAE Systems Image
--------------------------------------------------
Prompt: Best practices for online shopping
AI Response:
Best practices for online shopping in Japan

It is no secret that shopping in Japan is a great way to save money. There are lots of things that will save you money, so this article is focused on the most common items that you should be shopping for online.

Purchases of
--------------------------------------------------
Prompt: Benefits of using AI in business
AI Response:
Benefits of using AI in business

AI training can give you the skills you need to become a better business consultant. It includes:

Learning how to make

In [7]:
def demonstrate_content_and_classification():
    """Demo for sentiment analysis and text classification"""
    # --- Sentiment Analysis ---
    reviews = ["This product is amazing! Best purchase ever!","Terrible quality, waste of money.","It's okay, nothing special but does the job."]
    
    if 'sentiment' in ai_models:
        print("\nSentiment Analysis")
        for review in reviews:
            result = ai_models['sentiment'](review)[0]
            print(f"Review: {review}")
            print(f"Sentiment: {result['label']} | Confidence Score:{result['score']:.2f}\n")

# --- Text Classification ---

    texts = ["AI is transforming healthcare by improving diagnosis accuracy.","The football team won their championship game yesterday.","Investors are excited about the growth of renewable energy."]
    labels = ["Technology", "Sports", "Business", "Health"]
    if 'classifier' in ai_models:
        print("\nText Classification")
        for text in texts:
            result = ai_models['classifier'](text, candidate_labels=labels)
            print(f"Text: {text}")
            print(f"Category: {result['labels'][0]} | Confidence Score:{result['scores'][0]:.2f}\n")

# Run the demo
demonstrate_content_and_classification()


Sentiment Analysis
Review: This product is amazing! Best purchase ever!
Sentiment: POSITIVE | Confidence Score:1.00

Review: Terrible quality, waste of money.
Sentiment: NEGATIVE | Confidence Score:1.00

Review: It's okay, nothing special but does the job.
Sentiment: POSITIVE | Confidence Score:1.00


Text Classification
Text: AI is transforming healthcare by improving diagnosis accuracy.
Category: Technology | Confidence Score:0.64

Text: The football team won their championship game yesterday.
Category: Sports | Confidence Score:0.99

Text: Investors are excited about the growth of renewable energy.
Category: Business | Confidence Score:0.49



In [8]:
# Global variable to store workflow results
results = {}
def ai_collaboration():
    """Generator + Sentiment + Classifier working together"""
    product = "SmartWatch Pro"
    # Step 1: Generate description
    desc = ai_models['text_generator'](f"{product} features",max_new_tokens=20, temperature=0.7)[0]['generated_text']
    desc = desc.replace(f"{product} features", "").strip()
    full_text = f"{product} features {desc}"
    # Step 2: Sentiment analysis
    mood = ai_models['sentiment'](full_text)[0]
    sentiment = f"{mood['label']} ({mood['score']:.2f})"
    # Step 3: Classification
    categories = ["Electronics", "Sports", "Fashion"]
    cat = ai_models['classifier'](full_text, candidate_labels=categories)
    classification = f"{cat['labels'][0]} ({cat['scores'][0]:.2f})"
    # Save results
    results['ai_version'] = {"text": full_text,"sentiment": sentiment,"category": classification}

    # Print concise results
    print("Generated:", full_text)
    print("Sentiment:", sentiment)
    print("Category:", classification)
# Run the demo
ai_collaboration()

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated: SmartWatch Pro features a 6-inch 1080p LCD touchscreen with 16:9 resolution. This camera is connected via a
Sentiment: POSITIVE (0.64)
Category: Electronics (0.65)


In [9]:
# Step 2: Human feedback
def step2_human_feedback():
    """Collect human feedback on missing features"""
    original = results.get('ai_version', {}).get('text', 'SmartWatch Pro features basic functions')
    print("Original:", original)
    feedback = input("What features should be added? (e.g., battery life,waterproof): ")
    results['human_feedback'] = feedback.strip()
    return feedback

In [ ]:
# Step 3: AI improvement
def step3_direct_ai_improvement():
    """GPT-2 rewrites the description using human feedback"""
    original = results.get('ai_version', {}).get('text', '').strip()
    feedback = results.get('human_feedback', '').strip()
    # Short natural prompt (avoids repetition loops)
    prompt = (f"Original: {original}\n"f"Rewrite the product description to include {feedback}.\n")
    output = ai_models['text_generator'](prompt,max_new_tokens=120,temperature=0.7,top_p=0.9,pad_token_id=50256,return_full_text=False)[0]['generated_text']
    improved = output
    # Save for later use
    results['final_improved'] = improved
    print("Improved:", improved)
# Execute steps
step2_human_feedback()
step3_direct_ai_improvement()

Original: SmartWatch Pro features a 6-inch 1080p LCD touchscreen with 16:9 resolution. This camera is connected via a
